In [0]:
#get the file name from the adf
fileName = dbutils.widgets.get('fileName')
fileNameWithoutExt = fileName.split('.')[0]
print(fileNameWithoutExt)

Product


In [0]:
import pyspark.sql.functions as F
#from datetime import datetme as dt

#Just change all the values here based on the resource name you have created in your environemnt and workspace.

sqlDbName = 'apdb'
dbUserName = 'sdesql'
passwordKey = 'passwordkeydb'
stgAccountSASTokenKey = 'sasforaplanding'
landingFileName =fileName #'Product'  #dbutils.widgets.get('Product')
databricksScopeName ='approjectscope'
dbServer = 'dbmorgan'
dbServerPortNumber ='1433'
storageContainer ='input'
storageAccount='aplandingcsv'
landingMountPoint ='/mnt'


In [0]:
%python
# Get token from secret scope
sas_token = dbutils.secrets.get(
    scope=databricksScopeName,
    key=stgAccountSASTokenKey
)

# Configure direct ADLS Gen2 access without mounts
spark.conf.set(
    f"fs.azure.account.auth.type.{storageAccount}.dfs.core.windows.net",
    "SAS"
)
spark.conf.set(
    f"fs.azure.sas.token.provider.type.{storageAccount}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.sas.FixedSASTokenProvider"
)
spark.conf.set(
    f"fs.azure.sas.fixed.token.{storageAccount}.dfs.core.windows.net",
    sas_token
)

# Fix: point to the actual landing folder
landingBasePath = f"abfss://{storageContainer}@{storageAccount}.dfs.core.windows.net/landing"
rejectedBasePath = f"abfss://rejected@{storageAccount}.dfs.core.windows.net"
stagingBasePath = f"abfss://staging@{storageAccount}.dfs.core.windows.net"

# Validate access to the landing folder
print(dbutils.fs.ls(landingBasePath)[:5])

[FileInfo(path='abfss://input@aplandingcsv.dfs.core.windows.net/landing/Product.csv', name='Product.csv', size=56210, modificationTime=1787049881000)]


In [0]:
#connect to Azure SQL DB
dbPassword = dbutils.secrets.get(scope = databricksScopeName, key= passwordKey)
serverurl = 'jdbc:sqlserver://{}.database.windows.net:{};database={};user={};'.format(dbServer, dbServerPortNumber, 'apdb', dbUserName)
connectionProperties = {
    'password':dbPassword,
    'driver':'com.microsoft.sqlserver.jdbc.SQLServerDriver'
}
df = spark.read.jdbc(url = serverurl, table = 'dbo.FileDetailsFormat', properties= connectionProperties)
display(df)


FileNo,FileName,ColumnName,ColumnDateFormat,ColumnIsNull,ModifiedDate
1,Product,StartDate,MM-dd-yyyy,true,2012-06-18T22:34:09Z
1,Product,EndDate,MM/dd/yyyy,true,2012-06-18T22:34:09Z
1,Product,CreateDate,MM/dd/yyyy,true,2012-06-18T22:34:09Z
1,Product,ModifiedDate,MM/dd/yyyy,true,2012-06-18T22:34:09Z
2,ProductDescription,ModifiedDate,MM/dd/yyyy,true,2012-06-18T22:34:09Z
2,ProductDescription,StartDate,MM/dd/yyyy,true,2012-06-18T22:34:09Z
2,ProductDescription,EndDate,MM/dd/yyyy,true,2012-06-18T22:34:09Z
3,CustomerDetail,CreateDate,MM/dd/yyyy,true,2012-06-18T22:34:09Z
3,CustomerDetail,ActiveDate,MM/dd/yyyy,true,2012-06-18T22:34:09Z


In [0]:
if 'fileName' not in locals() or not fileName:
    fileName = dbutils.widgets.get('fileName')
if not fileName:
    raise ValueError("Set the fileName parameter before running Cell 5.")
if 'fileNameWithoutExt' not in locals() or not fileNameWithoutExt:
    fileNameWithoutExt = fileName.rsplit('.', 1)[0]

df1 = spark.read.csv(f"{landingBasePath}/{fileName}", inferSchema=True, header=True)
#display(df1)

# Rule
errorFlag=False
errorMessage = ''
totalcount = df1.count()
print(totalcount)
distinctCount = df1.distinct().count()
print(distinctCount)
if distinctCount !=totalcount:
    errorFlag = True
    errorMessage = 'Duplication Found. Rule 1 Failed'
print(errorMessage)
    
# Rule 2
df2 = df.filter(df.FileName==fileNameWithoutExt).select('ColumnName','ColumnDateFormat' )
rows = df2.collect()
for r in rows:
    colName = r[0]
    colFormat =r[1]
    print(colName, colFormat)
    #display(df1.filter(F.to_date(colName, colFormat).isNull() ==True))
    formatCount =df1.filter(F.to_date(F.col(colName), colFormat).isNotNull() ==True).count()
    if formatCount == totalcount:
        errorFlag = True
        errorMessage = errorMessage +' DateFormate is incorrect for {} '.format(colName)
    else:
        print('All rows are good for ', colName)
print(errorMessage)

rejectedTargetPath = f"abfss://{storageContainer}@{storageAccount}.dfs.core.windows.net/rejected/{fileName}"
stagingTargetPath = f"abfss://{storageContainer}@{storageAccount}.dfs.core.windows.net/staging/{fileName}"

if errorFlag:
    dbutils.fs.mv(f"{landingBasePath}/{fileName}", rejectedTargetPath)
    dbutils.notebook.exit('{"errorFlag": "true", "errorMessage":"'+errorMessage +'"}')
else:
    dbutils.fs.mv(f"{landingBasePath}/{fileName}", stagingTargetPath)
    dbutils.notebook.exit('{"errorFlag": "false", "errorMessage":"No error"}')